# 06_predictions_consolidation.ipynb
**DyPriZa – Consolidación final de predicciones para Power BI**

Este notebook genera el **CSV final** con precios *reales* y *predichos*, uniendo resultados de **Fase 05** y **Fase 05b**.
Se ejecuta **al final** del pipeline, justo antes de construir el dashboard de Power BI.

__Entradas__:
- `data/DyPriZa_predictions.csv` (Fase 05, modelo ganador)
- `data/DyPriZa_predictions_test_05b.csv` (Fase 05b, mejor modelo del grid)
- `data/DyPriZa_features.csv` (para predicción *full* opcional)
- `models/DyPriZa_best_model.joblib` (modelo MVP)

__Salidas__:
- `data/DyPriZa_predictions_final.csv`  → consolidado 05 + 05b
- `data/DyPriZa_predictions_summary.csv` → resumen de métricas por fuente/modelo
- `data/DyPriZa_predictions_full.csv`   → (opcional) predicción completa con el modelo MVP


## Parámetros

In [ ]:
from pathlib import Path 
from datetime import datetime 


DATA_DIR =Path ('data');DATA_DIR .mkdir (exist_ok =True )
MODEL_DIR =Path ('models');MODEL_DIR .mkdir (exist_ok =True )


MVP_MODEL_NAME ='LinearRegression'
MODEL_VERSION ='v1.0'


GENERAR_FULL =True 


## 1) Carga segura de entradas (05 y 05b)

In [ ]:
import pandas as pd 
import numpy as np 
import joblib 

P05 =DATA_DIR /'DyPriZa_predictions.csv'
P05B =DATA_DIR /'DyPriZa_predictions_test_05b.csv'
FEATURES =DATA_DIR /'DyPriZa_features.csv'
BEST_MODEL =MODEL_DIR /'DyPriZa_best_model.joblib'

def safe_read_csv (path ):
    try :
        df =pd .read_csv (path )
        print (f'✅ Cargado: {path} ->',df .shape )
        return df 
    except FileNotFoundError :
        print (f'⚠️ No encontrado: {path}')
        return pd .DataFrame ()

p05 =safe_read_csv (P05 )
p05b =safe_read_csv (P05B )
p05 .head (2 )if not p05 .empty else p05b .head (2 )


## 2) Consolidado final 05 + 05b → `DyPriZa_predictions_final.csv`

In [ ]:

if not p05 .empty :
    p05 =p05 .copy ();p05 ['modelo']=f'{MVP_MODEL_NAME} (Fase 05)'
if not p05b .empty :
    p05b =p05b .copy ();p05b ['modelo']='05b_best (GridSearch)'

consol =pd .concat ([p05 ,p05b ],axis =0 ,ignore_index =True )
if consol .empty :
    print ('⚠️ No hay datos de 05/05b para consolidar. Genera primero las predicciones.');consol 
else :

    cols_order =['precio_real','precio_predicho','error','fecha','propiedad','listing_id','property_id','modelo']
    consol =consol [[c for c in cols_order if c in consol .columns ]].copy ()

    consol ['modelo_final']=MVP_MODEL_NAME 
    consol ['version_modelo']=MODEL_VERSION 
    consol ['trained_at']=datetime .now ().strftime ('%Y-%m-%d %H:%M:%S')
    if 'fecha'in consol .columns :
        consol ['data_min_date']=consol ['fecha'].min ()
        consol ['data_max_date']=consol ['fecha'].max ()

    out_path =DATA_DIR /'DyPriZa_predictions_final.csv'
    consol .to_csv (out_path ,index =False )
    print ('💾 Guardado:',out_path ,consol .shape )
    consol .head (5 )


## 3) Resumen de métricas por fuente/modelo → `DyPriZa_predictions_summary.csv`

In [ ]:
def summarize (df ):
    if df .empty or 'precio_real'not in df .columns or 'precio_predicho'not in df .columns :
        return None 
    s =pd .Series (dtype =float )
    err =(df ['precio_real']-df ['precio_predicho']).astype (float )
    s ['MAE']=err .abs ().mean ()
    s ['RMSE']=np .sqrt (np .mean (np .square (err )))
    s ['R2']=1 -(np .sum (np .square (err ))/np .sum (np .square (df ['precio_real']-df ['precio_real'].mean ())))if df ['precio_real'].nunique ()>1 else np .nan 
    s ['N']=len (df )
    return s 

parts =[]
if not p05 .empty :
    s =summarize (p05 );s .name ='Fase 05 (MVP)';parts .append (s )
if not p05b .empty :
    s =summarize (p05b );s .name ='Fase 05b (best)';parts .append (s )

if parts :
    summary =pd .DataFrame (parts )
    summary_path =DATA_DIR /'DyPriZa_predictions_summary.csv'
    summary .to_csv (summary_path )
    print ('💾 Guardado:',summary_path )
    summary 
else :
    print ('⚠️ No hay datos para resumir. Revisa 05/05b.')


## 4) (Opcional) Predicción *full* con modelo MVP → `DyPriZa_predictions_full.csv`

In [ ]:
if GENERAR_FULL :
    if BEST_MODEL .exists ()and FEATURES .exists ():
        model =joblib .load (BEST_MODEL )
        df =pd .read_csv (FEATURES )
        drop_cols =[c for c in ['fecha','propiedad','listing_id','property_id','precio','price','tarifa','adr']if c in df .columns ]
        X =df .drop (columns =drop_cols ,errors ='ignore').select_dtypes (include =[np .number ])
        y_hat =model .predict (X )
        out =pd .DataFrame ({'precio_predicho':y_hat })

        target =next ((t for t in ['precio','price','tarifa','adr']if t in df .columns ),None )
        if target :
            out ['precio_real']=pd .to_numeric (df [target ],errors ='coerce')
            out ['error']=out ['precio_real']-out ['precio_predicho']

        for idcol in ['fecha','propiedad','listing_id','property_id']:
            if idcol in df .columns :
                out [idcol ]=df [idcol ]
        out_path =DATA_DIR /'DyPriZa_predictions_full.csv'
        out .to_csv (out_path ,index =False )
        print ('💾 Guardado:',out_path ,out .shape )
        out .head (5 )
    else :
        print ('ℹ️ Se omitió la predicción full: falta el modelo o el features.')
